<a href="https://colab.research.google.com/github/Jpedrogama/introducao-data-science/blob/master/DIO_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Default title text
import pandas as pd
import numpy as np
# Tipo de algortimo de classificação
from sklearn.ensemble import RandomForestClassifier
# Função para separar o banco de dados de treino e validação
from sklearn.model_selection import train_test_split

In [ ]:
# Montar dataset no pandas
data = pd.read_csv('/content/train.csv')
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# Função drop, para tirar da tabela os dados que não são relevantes para nosso estudo da base. Exemplo, nome, ticket o porto que embarcou...
data = data.drop(['Name', 'Ticket', 'Embarked'], axis = 1)
data.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin
0,1,0,3,male,22.0,1,0,7.2500,NaN
1,2,1,1,female,38.0,1,0,71.2833,C85
2,3,1,3,female,26.0,0,0,7.9250,NaN
3,4,1,1,female,35.0,1,0,53.1000,C123
4,5,0,3,male,35.0,0,0,8.0500,NaN


In [ ]:
# Define a chave primaria do dataframe sendo o PassengerId
data = data.set_index(['PassengerId'])
# Renomeia uma coluna do dataset
data = data.rename(columns= {'Survived': 'Target'}, inplace= False)
data.head()

,Target,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin
PassengerId,,,,,,,,
1,0,3,male,22.0,1,0,7.2500,NaN
2,1,1,female,38.0,1,0,71.2833,C85
3,1,3,female,26.0,0,0,7.9250,NaN
4,1,1,female,35.0,1,0,53.1000,C123
5,0,3,male,35.0,0,0,8.0500,NaN


In [ ]:
# Faz com que todos os dados numericos são metrificados e retornados em uma tabela. 
#Assim conseguimos saber os números que podemos usar para usar nas nossas estatisticas e modelos.  
data.describe()

,Target,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [ ]:
# No describe, pegamos os dados categoricos, mas que não fazem sentido fazer média, e ver outras informações estatisticas
# Mas podemos passar o include['O'] para que possamos ter noção de como nossos dados estão distribuidos
data.describe(include=['O'])

,Sex,Cabin
count,891,204
unique,2,147
top,male,B96 B98
freq,577,4


In [ ]:
# Para transformar os dados categoricos em dados, e adaptar para poder ser mais fácilmente aceito por algoritmos de machine learning, 
# Vamos trabalhar com as transformações
# Feito um processo manual para primeiro projeto, mas existem ferramentas que fazem isso de forma mais automatizada

# Se 'Sex' for female, converte para 1, se não for, converte para zero.
data['is_Woman'] = np.where(data['Sex'] == 'female', 1, 0)

# Convertendo as classes (primeira classe, segunda classe e terceira classe) em três tabelas diferentes com 1 para sim e 0 para não.
data['Pclass_1'] = np.where(data['Pclass'] ==1, 1, 0)
data['Pclass_2'] = np.where(data['Pclass'] ==2, 1, 0)
data['Pclass_3'] = np.where(data['Pclass'] ==3, 1, 0)

# Tira os dados que ficam duplicados (a coluna Pclass e a coluna do sexo como dado categorico)
data = data.drop(['Sex', 'Pclass'], axis=1)
data.head()

,Target,Age,SibSp,Parch,Fare,Cabin,is_Woman,Pclass_1,Pclass_2,Pclass_3
PassengerId,,,,,,,,,,
1,0,22.0,1,0,7.2500,NaN,0,0,0,1
2,1,38.0,1,0,71.2833,C85,1,1,0,0
3,1,26.0,0,0,7.9250,NaN,1,0,0,1
4,1,35.0,1,0,53.1000,C123,1,1,0,0
5,0,35.0,0,0,8.0500,NaN,0,0,0,1


In [ ]:
# Verificar os dados que estão vazios na nossa tabela
# Verificamos que existem 177 pessoas sem o dado de idade categorizado
# E 687 Sem a cabine definida
data.isnull().sum()

Target        0
Age         177
SibSp         0
Parch         0
Fare          0
Cabin       687
is_Woman      0
Pclass_1      0
Pclass_2      0
Pclass_3      0
dtype: int64

In [ ]:
# Nesse caso, vamos subistituir todos os valores nulos por 0.
data.fillna(0, inplace=True)
data.isnull().sum()

Target      0
Age         0
SibSp       0
Parch       0
Fare        0
Cabin       0
is_Woman    0
Pclass_1    0
Pclass_2    0
Pclass_3    0
dtype: int64

In [ ]:
# Para o algoritmo da amostragem, vamos separar a base de dados em treino e teste.
# Com isso, vamos limpar algumas informações 
# Separo com 30% (test_size) para o teste e 70% para treinamento
x_train, x_test, y_train, y_test = train_test_split(
    data.drop(['Target', 'Cabin'], axis=1),
    data['Target'],
    test_size = 0.3,
    random_state = 1234
)

[{'train': x_train.shape}, {'test': x_test.shape}]

[{'train': (623, 8)}, {'test': (268, 8)}]

In [ ]:
# Modelo de floresta aleatoria:
# Sobre arvores de decisão (prévia para random forest)
# https://www.youtube.com/watch?v=_L39rN6gz7Y&ab_channel=StatQuestwithJoshStarmer
# Sobre random Forest
# https://www.youtube.com/watch?v=J4Wdy0Wc_xQ
# https://www.youtube.com/watch?v=nyxTdL_4Q-Q

In [ ]:
rndforest = RandomForestClassifier(n_estimators = 1000,
                                   criterion= 'gini',
                                   max_depth = 5)

rndforest.fit(x_train, y_train)

RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=5, max_features='auto',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=1000,
                       n_jobs=None, oob_score=False, random_state=None,
                       verbose=0, warm_start=False)

In [ ]:
probability = rndforest.predict_proba(data.drop(['Target', 'Cabin'], axis=1))[:,1]
classification = rndforest.predict(data.drop(['Target','Cabin' ], axis=1))

In [ ]:
data['probability'] = probability
data['classification'] = classification

In [ ]:
data

,Target,Age,SibSp,Parch,Fare,Cabin,is_Woman,Pclass_1,Pclass_2,Pclass_3,probability,classification
PassengerId,,,,,,,,,,,,
1,0,22.0,1,0,7.2500,0,0,0,0,1,0.121955,0
2,1,38.0,1,0,71.2833,C85,1,1,0,0,0.935647,1
3,1,26.0,0,0,7.9250,0,1,0,0,1,0.442209,0
4,1,35.0,1,0,53.1000,C123,1,1,0,0,0.917329,1
5,0,35.0,0,0,8.0500,0,0,0,0,1,0.139403,0
...,...,...,...,...,...,...,...,...,...,...,...,...
887,0,27.0,0,0,13.0000,0,0,0,1,0,0.128179,0
888,1,19.0,0,0,30.0000,B42,1,1,0,0,0.848094,1
889,0,0.0,1,2,23.4500,0,1,0,0,1,0.477165,0
